# Preprocessing Lokal – Dataset Klasifikasi Sampah (SigLIP)

**Alur:**
1. Tahap 1 – Deduplikasi Gambar (pixel hash SHA256 + MD5 file)
2. Tahap 2 – Filter Kualitas & Resolusi + Konversi ke RGB
3. Tahap 3 – Hitung Class Weights & Catat Rencana Augmentasi
4. Tahap 4 – Stratified Split train/val/test (70/15/15)

**Deliverable:**
- `dedup_log.csv` – daftar duplikat yang dipindahkan
- `metadata_kualitas.csv` – metadata gambar + flag kualitas
- `class_weights.json` – bobot kelas untuk loss function
- `train.csv`, `val.csv`, `test.csv` – manifest split

In [ ]:
# ============================================================
# KONFIGURASI – Ganti ROOT_DIR sesuai lokasi folder dataset
# ============================================================
import os

ROOT_DIR = r"d:\UGM\Semester 5\Senpro\Project\Senpro"   # root project
DATASET_DIR = os.path.join(ROOT_DIR, "Dataset")           # folder berisi 0_Recylable, 1_Organik, 2_B3
OUTPUT_DIR = os.path.join(ROOT_DIR, "preprocessing_output") # hasil output semua file CSV/JSON

# Label mapping: nama folder -> nama kelas
CLASS_MAP = {
    "0_Recylable": "Recylable",
    "1_Organik": "Organik",
    "2_B3": "B3",
}

# Ambang resolusi minimum
MIN_RESOLUTION = 50   # piksel (lebar DAN tinggi harus >= nilai ini)

# Threshold deteksi kecerahan
DARK_THRESHOLD   = 50   # rata-rata brightness (0-255) di bawah ini → terindikasi gelap
BRIGHT_THRESHOLD = 220  # rata-rata brightness di atas ini → terindikasi overexposed

# Split ratio
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

RANDOM_SEED = 42

# ------------------------------------------------------------
# TAHAP 1.5 – Data Cleaning dari review embedding (opsional)
# ------------------------------------------------------------
# CSV hasil Cell 12b notebook probe (data_diagnostics/<embedding>/label_noise_candidates.csv)
# yang kolom 'keputusan'-nya SUDAH diisi manual: 'buang' | 'relabel:<Kelas>' | 'ok' (kosong = lewati).
# Bisa daftar beberapa CSV (mis. hasil review B3 + hasil review Recylable/Organik).
# None atau file tidak ada -> Tahap 1.5 dilewati (tidak menyentuh apa pun).
CLEANING_CSVS = [
    os.path.join(ROOT_DIR, "siglip-so400m-patch14-384_naug_2_2_5",
                 "review_RO_label_noise_cleaned.csv"),
]
# Kelas valid untuk relabel (ejaan 'Recylable' disengaja)
VALID_LABELS = set(CLASS_MAP.values())

# Ambang Hamming dHash untuk near-duplicate perseptual di Tahap 1 (0 = identik; 5 ~ sangat mirip)
DHASH_MAX_HAMMING = 5

# CSV near-duplicate lintas split dari Cell 12b notebook probe
# (data_diagnostics/<embedding>/cross_split_near_duplicates.csv). Gambar val/test yang punya
# kembaran di train akan DIPINDAH ke TRAIN saat split -> val/test bebas bayangan train.
# None / file tidak ada -> langkah ini dilewati.
CROSS_SPLIT_DUP_CSV = None

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"[OK] OUTPUT_DIR: {OUTPUT_DIR}")
print(f"[OK] DATASET_DIR: {DATASET_DIR}")

In [2]:
# ============================================================
# INSTALASI DEPENDENSI (jalankan sekali jika belum ada)
# ============================================================
# %pip install tqdm pillow pandas scikit-learn

In [3]:
# ============================================================
# IMPORTS
# ============================================================
import hashlib
import json
import shutil
from pathlib import Path
from collections import defaultdict

import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split

print("[OK] Semua library berhasil diimport.")

[OK] Semua library berhasil diimport.


---
## TAHAP 1 – Deduplikasi Gambar

In [ ]:
def compute_md5(filepath: Path) -> str:
    """Hitung MD5 hash dari byte file mentah."""
    h = hashlib.md5()
    with open(filepath, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()


def compute_pixel_sha256(filepath: Path) -> str | None:
    """
    Hitung SHA256 dari kombinasi (lebar, tinggi, pixel_bytes_RGB).
    Return None jika gambar tidak bisa dibuka.
    """
    try:
        with Image.open(filepath) as img:
            img_rgb = img.convert('RGB')
            w, h = img_rgb.size
            pixel_bytes = img_rgb.tobytes()
        h_sha = hashlib.sha256()
        h_sha.update(w.to_bytes(4, 'little'))
        h_sha.update(h.to_bytes(4, 'little'))
        h_sha.update(pixel_bytes)
        return h_sha.hexdigest()
    except Exception:
        return None


def compute_dhash(filepath: Path, hash_size: int = 8) -> str | None:
    """Perceptual hash (difference hash). Tahan terhadap re-encode / beda format
    (mis. .png vs .jpg dari gambar yang sama) yang lolos MD5/SHA256 piksel.
    Grayscale -> resize (hash_size+1 x hash_size) -> bandingkan piksel bertetangga -> bit.
    Return hex string, atau None bila gambar tak terbaca."""
    try:
        with Image.open(filepath) as img:
            small = img.convert('L').resize((hash_size + 1, hash_size),
                                            Image.Resampling.LANCZOS)
        px = list(small.getdata())
        bits = 0
        for row in range(hash_size):
            for col in range(hash_size):
                left  = px[row * (hash_size + 1) + col]
                right = px[row * (hash_size + 1) + col + 1]
                bits = (bits << 1) | int(left > right)
        return format(bits, '0{}x'.format(hash_size * hash_size // 4))
    except Exception:
        return None


def hamming_hex(a: str, b: str) -> int:
    """Jarak Hamming antara dua dHash hex (jumlah bit berbeda)."""
    return bin(int(a, 16) ^ int(b, 16)).count('1')


def collect_all_images(dataset_dir: str, class_map: dict) -> list[dict]:
    """
    Kumpulkan semua file gambar dari seluruh subfolder (rekursif).
    Return list of dict {path, rel_path, label, filename}.
    """
    valid_ext = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.gif', '.tiff'}
    records = []
    for folder_name, label in class_map.items():
        class_dir = Path(dataset_dir) / folder_name
        if not class_dir.exists():
            print(f"[WARN] Folder tidak ditemukan: {class_dir}")
            continue
        # rglob → rekursif ke semua subfolder
        for fpath in sorted(class_dir.rglob('*')):
            if fpath.suffix.lower() in valid_ext and fpath.is_file():
                rel_path = fpath.relative_to(dataset_dir)
                records.append({
                    'path': fpath,
                    'rel_path': str(rel_path),
                    'label': label,
                    'filename': fpath.name,
                    'class_folder': folder_name,
                })
    return records


print("[INFO] Mengumpulkan semua gambar...")
all_images = collect_all_images(DATASET_DIR, CLASS_MAP)
print(f"[OK] Total gambar ditemukan: {len(all_images)}")

# Ringkasan per kelas
from collections import Counter
label_counts = Counter(r['label'] for r in all_images)
for label, cnt in sorted(label_counts.items()):
    print(f"  {label}: {cnt} gambar")

In [ ]:
# Hitung hash untuk setiap gambar
print("[INFO] Menghitung signature gambar (MD5 + SHA256 pixel + dHash perseptual)...")
for rec in tqdm(all_images, desc="Hashing", unit="img"):
    rec['md5']         = compute_md5(rec['path'])
    rec['pixel_sha256'] = compute_pixel_sha256(rec['path'])
    rec['dhash']        = compute_dhash(rec['path'])

print("[OK] Selesai hashing.")

In [ ]:
def find_duplicates(records: list[dict], dhash_max_hamming: int = 5) -> tuple[set, list[dict]]:
    """
    Deteksi duplikat berdasarkan union MD5, pixel_sha256, dan dHash perseptual
    (dHash menangkap gambar sama yang di-reencode / beda format: beda hash eksak, mirip visual).
    Untuk setiap grup duplikat, pilih representatif terbaik:
      - Prioritas: resolusi tertinggi (lebar * tinggi)
      - Tie-break: nama file terpendek, lalu alfabetis
    Return:
      - keep_paths: set of path string untuk file yang DIPERTAHANKAN
      - dup_log: list of dict untuk file yang DIHAPUS (dipindahkan)
    """
    # Kelompokkan berdasarkan masing-masing hash
    md5_groups   = defaultdict(list)
    sha256_groups = defaultdict(list)

    for rec in records:
        if rec['md5']:
            md5_groups[rec['md5']].append(rec)
        if rec['pixel_sha256']:
            sha256_groups[rec['pixel_sha256']].append(rec)

    # Kumpulkan semua grup duplikat (union)
    # Gunakan path sebagai identifier
    dup_groups: list[list[dict]] = []
    
    def get_resolution(rec):
        try:
            with Image.open(rec['path']) as img:
                return img.width * img.height
        except Exception:
            return 0

    def pick_representative(group: list[dict]) -> dict:
        """Pilih gambar dengan resolusi terbesar; tie-break: nama file pendek lalu alfa."""
        return max(
            group,
            key=lambda r: (get_resolution(r), -len(r['filename']), r['filename'])
        )

    # Proses MD5 duplikat
    md5_dup_paths: set[str] = set()  # path yang sudah masuk grup
    for h, group in md5_groups.items():
        if len(group) < 2:
            continue
        dup_groups.append(group)
        for r in group:
            md5_dup_paths.add(str(r['path']))

    # Proses SHA256 duplikat (tambahkan grup baru yang belum tercakup MD5)
    sha256_dup_paths: set[str] = set()
    for h, group in sha256_groups.items():
        if len(group) < 2:
            continue
        group_paths = {str(r['path']) for r in group}
        # Hanya tambahkan jika ada path yang belum tercakup MD5 group
        if not group_paths.issubset(md5_dup_paths):
            dup_groups.append(group)
            for p in group_paths:
                sha256_dup_paths.add(p)

    # Proses dHash perseptual: kelompokkan dalam KELAS yang sama (hindari gabung lintas kelas),
    # pakai bucket exact-dHash lalu gabung bucket yang berjarak Hamming <= ambang.
    covered = md5_dup_paths | sha256_dup_paths
    by_class = defaultdict(list)
    for r in records:
        if r.get('dhash'):
            by_class[r['label']].append(r)
    for label, recs in by_class.items():
        buckets = defaultdict(list)                       # dhash exact -> records
        for r in recs:
            buckets[r['dhash']].append(r)
        keys = list(buckets)
        used = set()
        for i, hk in enumerate(keys):
            if hk in used:
                continue
            group_keys = [hk]
            for hk2 in keys[i + 1:]:
                if hk2 not in used and hamming_hex(hk, hk2) <= dhash_max_hamming:
                    group_keys.append(hk2); used.add(hk2)
            group = [r for k in group_keys for r in buckets[k]]
            if len(group) < 2:
                continue
            gpaths = {str(r['path']) for r in group}
            if gpaths.issubset(covered):                  # sudah tertangani MD5/SHA256
                continue
            dup_groups.append(group)
            covered |= gpaths

    # Tentukan yang dipertahankan per grup, sisanya → hapus/pindahkan
    path_to_rec = {str(r['path']): r for r in records}
    to_remove: set[str] = set()
    dup_log: list[dict] = []

    for group in dup_groups:
        rep = pick_representative(group)
        for r in group:
            if str(r['path']) != str(rep['path']) and str(r['path']) not in to_remove:
                to_remove.add(str(r['path']))
                dup_log.append({
                    'file_dihapus': r['rel_path'],
                    'label': r['label'],
                    'file_representatif': rep['rel_path'],
                    'md5': r['md5'],
                    'pixel_sha256': r['pixel_sha256'] or '',
                    'alasan': ('md5_match' if r['md5'] == rep['md5']
                               else 'pixel_match' if r.get('pixel_sha256') and r['pixel_sha256'] == rep.get('pixel_sha256')
                               else 'dhash_match'),
                })

    keep_paths = {str(r['path']) for r in records} - to_remove
    return keep_paths, dup_log


print("[INFO] Mendeteksi duplikat...")
keep_paths, dup_log = find_duplicates(all_images, dhash_max_hamming=DHASH_MAX_HAMMING)
print(f"[OK] Duplikat ditemukan: {len(dup_log)} file")
print(f"[OK] File dipertahankan: {len(keep_paths)} file")

In [7]:
# Pindahkan duplikat ke folder duplicates_removed
DUPLICATES_DIR = Path(DATASET_DIR) / "duplicates_removed"
DUPLICATES_DIR.mkdir(exist_ok=True)

print(f"[INFO] Memindahkan {len(dup_log)} duplikat ke: {DUPLICATES_DIR}")
move_errors = []
for entry in tqdm(dup_log, desc="Memindahkan duplikat", unit="file"):
    src = Path(DATASET_DIR) / entry['file_dihapus']
    # Pertahankan struktur subfolder di dalam duplicates_removed
    dst = DUPLICATES_DIR / entry['file_dihapus']
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.exists():
        try:
            shutil.move(str(src), str(dst))
        except Exception as e:
            move_errors.append({'file': entry['file_dihapus'], 'error': str(e)})
    else:
        move_errors.append({'file': entry['file_dihapus'], 'error': 'source_not_found'})

if move_errors:
    print(f"[WARN] {len(move_errors)} file gagal dipindahkan.")
else:
    print("[OK] Semua duplikat berhasil dipindahkan.")

[INFO] Memindahkan 0 duplikat ke: d:\UGM\Semester 5\Senpro\Project\Senpro\Dataset\duplicates_removed


Memindahkan duplikat: 0file [00:00, ?file/s]

Memindahkan duplikat: 0file [00:00, ?file/s]

[OK] Semua duplikat berhasil dipindahkan.


In [8]:
# Simpan dedup_log.csv
dedup_log_path = os.path.join(OUTPUT_DIR, "dedup_log.csv")
df_dedup = pd.DataFrame(dup_log)
df_dedup.to_csv(dedup_log_path, index=False)
print(f"[OK] dedup_log.csv disimpan: {dedup_log_path}")

# Laporan per kelas
print("\n=== LAPORAN DEDUPLIKASI ===")
before_counts = Counter(r['label'] for r in all_images)
kept_records  = [r for r in all_images if str(r['path']) in keep_paths]
after_counts  = Counter(r['label'] for r in kept_records)
for label in sorted(before_counts.keys()):
    b = before_counts[label]
    a = after_counts.get(label, 0)
    print(f"  {label}: {b} → {a} (dihapus: {b - a})")
print(f"  TOTAL: {sum(before_counts.values())} → {sum(after_counts.values())}")

[OK] dedup_log.csv disimpan: d:\UGM\Semester 5\Senpro\Project\Senpro\preprocessing_output\dedup_log.csv

=== LAPORAN DEDUPLIKASI ===
  B3: 3637 → 3637 (dihapus: 0)
  Organik: 13744 → 13744 (dihapus: 0)
  Recylable: 12944 → 12944 (dihapus: 0)
  TOTAL: 30325 → 30325


---
## TAHAP 1.5 – Data Cleaning dari Review Embedding (opsional)

Menerapkan keputusan review manual kandidat *label noise* (dari Cell 12b `siglip_embedding_probe_colab.ipynb`)
**secara permanen**, sebelum filter kualitas & split. Jalur ini satu-satunya yang menggerakkan metrik secara
berarti (semua head sudah ~99%; sisa error di pasangan **Recylable↔Organik**, B3 sudah sangat baik).

Format CSV (`CLEANING_CSVS` di Cell konfigurasi) wajib punya kolom `path` dan `keputusan`:

| keputusan | arti | tindakan fisik |
|---|---|---|
| `buang` | sampel rusak/off-distribution | pindah ke `Dataset/duplicates_removed/` (tidak dihapus) |
| `relabel:<Kelas>` | label salah | pindah ke folder kelas tujuan (`0_Recylable`/`1_Organik`/`2_B3`) |
| `ok` / kosong | benar | tidak diubah |

Aman & idempoten: file yang sudah tidak ada di lokasi asal dilewati; tidak ada yang dihapus permanen.
Hasil dicatat ke `cleaning_log.csv`. Semua perubahan dibaca ulang oleh Tahap 2 (karena memakai hasil
`collect_all_images` yang diperbarui), lalu split pada Tahap 4 memakai data yang sudah bersih.


In [ ]:
def _rel_to_abs(rel_path: str) -> Path:
    return Path(DATASET_DIR) / str(rel_path).replace('/', os.sep)


def _match_key(p) -> str:
    """Kunci cocok = '<folder_kelas>/<nama_file>' (tahan beda root/separator)."""
    parts = str(p).replace('\\', '/').split('/')
    return '/'.join(parts[-2:])


def apply_cleaning(csv_paths, kept_records):
    """Terapkan keputusan review. Mengubah file di disk + mengembalikan daftar record train yang
    sudah bersih (dengan label terbaru) untuk Tahap 2."""
    # index record saat ini berdasarkan match-key
    by_key = {_match_key(r['rel_path']): r for r in kept_records}
    decisions = {}        # match_key -> ('buang', None) | ('relabel', '2_B3')
    folder_of = {v: k for k, v in CLASS_MAP.items()}   # 'B3' -> '2_B3'

    n_csv = 0
    for cp in (csv_paths or []):
        if not cp or not os.path.exists(cp):
            continue
        n_csv += 1
        df = pd.read_csv(cp)
        if 'path' not in df.columns or 'keputusan' not in df.columns:
            print(f"  [WARN] {os.path.basename(cp)}: butuh kolom 'path' & 'keputusan' -> dilewati.")
            continue
        for _, row in df.iterrows():
            kep = str(row['keputusan']).strip().lower()
            key = _match_key(row['path'])
            if kep in ('', 'ok', 'nan'):
                continue                                   # label asli benar / belum diputuskan -> lewati
            if kep.startswith('buang'):
                decisions[key] = ('buang', None)
                continue
            # 'setuju' = setujui usulan OOF di kolom 'saran_relabel' (mis. 'relabel:Organik')
            if kep.startswith('setuju'):
                src_dec = str(row.get('saran_relabel', '')).strip()
            elif kep.startswith('relabel'):
                src_dec = str(row['keputusan']).strip()
            else:
                print(f"  [WARN] keputusan tak dikenal '{row['keputusan']}' untuk {key} -> dilewati.")
                continue
            tgt = src_dec.split(':', 1)[1].strip() if ':' in src_dec else ''
            if tgt not in VALID_LABELS:
                print(f"  [WARN] target relabel tak valid '{src_dec}' untuk {key} -> dilewati.")
                continue
            decisions[key] = ('relabel', folder_of[tgt])

    if n_csv == 0:
        print('[SKIP] Tahap 1.5: tidak ada CSV cleaning (CLEANING_CSVS kosong / file belum ada).')
        return kept_records, []

    dup_dir = Path(DATASET_DIR) / 'duplicates_removed'
    log, n_buang, n_relabel = [], 0, 0
    for key, (aksi, tgt_folder) in decisions.items():
        rec = by_key.get(key)
        if rec is None:
            log.append({'match_key': key, 'aksi': aksi, 'status': 'tidak_ditemukan_di_dataset'})
            continue
        src = _rel_to_abs(rec['rel_path'])
        if not src.exists():
            log.append({'match_key': key, 'aksi': aksi, 'status': 'file_hilang'})
            continue

        if aksi == 'buang':
            dst = dup_dir / rec['rel_path']
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(src), str(dst))
            rec['_removed'] = True
            n_buang += 1
            log.append({'match_key': key, 'aksi': 'buang', 'label_lama': rec['label'],
                        'label_baru': '', 'tujuan': str(dst.relative_to(DATASET_DIR)), 'status': 'ok'})
        else:  # relabel
            new_label = CLASS_MAP[tgt_folder]
            dst = Path(DATASET_DIR) / tgt_folder / rec['filename']
            k = 1
            while dst.exists():                              # hindari menimpa nama yang sama
                dst = Path(DATASET_DIR) / tgt_folder / f'{src.stem}_relabel{k}{src.suffix}'
                k += 1
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(src), str(dst))
            rec['label'] = new_label
            rec['class_folder'] = tgt_folder
            rec['rel_path'] = str(dst.relative_to(DATASET_DIR)).replace(os.sep, '/')
            rec['path'] = dst
            rec['filename'] = dst.name
            n_relabel += 1
            log.append({'match_key': key, 'aksi': 'relabel', 'label_lama': '',
                        'label_baru': new_label, 'tujuan': rec['rel_path'], 'status': 'ok'})

    cleaned = [r for r in kept_records if not r.get('_removed')]
    df_log = pd.DataFrame(log)
    log_path = os.path.join(OUTPUT_DIR, 'cleaning_log.csv')
    df_log.to_csv(log_path, index=False)

    print(f'[OK] Tahap 1.5 selesai: {n_relabel} relabel, {n_buang} dibuang '
          f'(dari {len(decisions)} keputusan di {n_csv} CSV).')
    print(f'     Record train bersih: {len(kept_records)} -> {len(cleaned)}')
    print(f'     Log: {log_path}')
    print('\n=== DISTRIBUSI KELAS SETELAH CLEANING ===')
    for label, cnt in sorted(Counter(r['label'] for r in cleaned).items()):
        print(f'  {label}: {cnt}')
    return cleaned, log


kept_records, cleaning_log = apply_cleaning(CLEANING_CSVS, kept_records)


---
## TAHAP 2 – Filter Kualitas & Resolusi + Konversi RGB

In [9]:
def analyze_image_quality(filepath: Path, dark_thresh: int, bright_thresh: int) -> dict:
    """
    Analisis kualitas gambar: status, resolusi, mode warna, flag kecerahan.
    Jika gambar bukan RGB, konversi dan simpan ulang.
    Return dict dengan hasil analisis.
    """
    result = {
        'status': 'ok',
        'lebar': None,
        'tinggi': None,
        'mode_awal': None,
        'dikonversi_ke_rgb': False,
        'terindikasi_gelap': False,
        'terindikasi_overexposed': False,
        'rata_kecerahan': None,
        'error': '',
    }
    try:
        with Image.open(filepath) as img:
            result['mode_awal'] = img.mode
            result['lebar']    = img.width
            result['tinggi']   = img.height

            # Filter resolusi minimum
            if img.width < MIN_RESOLUTION or img.height < MIN_RESOLUTION:
                result['status'] = 'dropped_low_res'
                return result

            # Konversi ke RGB jika perlu
            if img.mode != 'RGB':
                img_rgb = img.convert('RGB')
                result['dikonversi_ke_rgb'] = True
                # Simpan ulang menimpa file asli dengan format yang sama
                suffix = filepath.suffix.lower()
                fmt_map = {'.jpg': 'JPEG', '.jpeg': 'JPEG', '.png': 'PNG',
                           '.bmp': 'BMP', '.webp': 'WEBP', '.tiff': 'TIFF'}
                save_fmt = fmt_map.get(suffix, 'JPEG')
                img_rgb.save(str(filepath), format=save_fmt)
            else:
                img_rgb = img.copy()

            # Hitung rata-rata kecerahan (grayscale)
            gray = img_rgb.convert('L')
            arr  = np.array(gray, dtype=np.float32)
            mean_brightness = float(arr.mean())
            result['rata_kecerahan'] = round(mean_brightness, 2)

            # Flag kecerahan — tidak dibuang, hanya ditandai
            if mean_brightness < dark_thresh:
                result['terindikasi_gelap'] = True
            elif mean_brightness > bright_thresh:
                result['terindikasi_overexposed'] = True

    except Exception as e:
        result['status'] = 'corrupt'
        result['error']  = str(e)

    return result


print("[INFO] Menganalisis kualitas gambar (ini bisa memakan waktu)...")
quality_records = []
for rec in tqdm(kept_records, desc="Filter kualitas", unit="img"):
    q = analyze_image_quality(rec['path'], DARK_THRESHOLD, BRIGHT_THRESHOLD)
    quality_records.append({
        'rel_path': rec['rel_path'],
        'label': rec['label'],
        'filename': rec['filename'],
        **q,
    })

print("[OK] Analisis kualitas selesai.")

[INFO] Menganalisis kualitas gambar (ini bisa memakan waktu)...


Filter kualitas:   0%|          | 0/30325 [00:00<?, ?img/s]

Filter kualitas:   0%|          | 103/30325 [00:00<00:29, 1024.40img/s]

Filter kualitas:   1%|          | 206/30325 [00:00<00:40, 740.58img/s] 

Filter kualitas:   1%|          | 285/30325 [00:00<00:42, 699.82img/s]

...

Filter kualitas:  99%|█████████▉| 30129/30325 [00:25<00:00, 605.32img/s] 

Filter kualitas: 100%|█████████▉| 30230/30325 [00:25<00:00, 514.56img/s]

Filter kualitas: 100%|█████████▉| 30311/30325 [00:25<00:00, 484.36img/s]

Filter kualitas: 100%|██████████| 30325/30325 [00:25<00:00, 1182.00img/s]

[OK] Analisis kualitas selesai.


In [10]:
df_quality = pd.DataFrame(quality_records)

# Gambar yang lulus filter (status ok)
df_clean = df_quality[df_quality['status'] == 'ok'].copy()
df_dropped = df_quality[df_quality['status'] != 'ok'].copy()

print("=== LAPORAN FILTER KUALITAS ===")
print(f"  Total sebelum filter : {len(df_quality)}")
print(f"  Status 'ok'          : {len(df_clean)}")
print(f"  Corrupt              : {(df_quality['status'] == 'corrupt').sum()}")
print(f"  Resolusi terlalu kecil: {(df_quality['status'] == 'dropped_low_res').sum()}")
print(f"  Dikonversi ke RGB    : {df_quality['dikonversi_ke_rgb'].sum()}")
print(f"  Terindikasi gelap    : {df_clean['terindikasi_gelap'].sum()}")
print(f"  Terindikasi overexposed: {df_clean['terindikasi_overexposed'].sum()}")

# Simpan metadata kualitas
metadata_path = os.path.join(OUTPUT_DIR, "metadata_kualitas.csv")
df_quality.to_csv(metadata_path, index=False)
print(f"\n[OK] metadata_kualitas.csv disimpan: {metadata_path}")

# Per kelas
print("\n=== DISTRIBUSI KELAS SETELAH FILTER ===")
for label, cnt in sorted(df_clean['label'].value_counts().items()):
    print(f"  {label}: {cnt}")

=== LAPORAN FILTER KUALITAS ===
  Total sebelum filter : 30325
  Status 'ok'          : 30154
  Corrupt              : 0
  Resolusi terlalu kecil: 171
  Dikonversi ke RGB    : 0
  Terindikasi gelap    : 198
  Terindikasi overexposed: 4900

[OK] metadata_kualitas.csv disimpan: d:\UGM\Semester 5\Senpro\Project\Senpro\preprocessing_output\metadata_kualitas.csv

=== DISTRIBUSI KELAS SETELAH FILTER ===
  B3: 3466
  Organik: 13744
  Recylable: 12944


---
## TAHAP 3 – Class Weights & Rencana Augmentasi

In [11]:
# ============================================================
# HITUNG CLASS WEIGHTS (inversely proportional)
# Formula: weight_c = N_total / (N_classes * N_c)
# Ini adalah formula sklearn class_weight='balanced'
# ============================================================
class_counts = df_clean['label'].value_counts().to_dict()
n_total      = sum(class_counts.values())
n_classes    = len(class_counts)

class_weights = {
    label: round(n_total / (n_classes * count), 6)
    for label, count in class_counts.items()
}

# Simpan sebagai JSON
cw_path = os.path.join(OUTPUT_DIR, "class_weights.json")
with open(cw_path, 'w') as f:
    json.dump(class_weights, f, indent=2)

print("=== CLASS WEIGHTS ===")
for label, w in sorted(class_weights.items()):
    print(f"  {label}: count={class_counts[label]}, weight={w:.4f}")
print(f"\n[OK] class_weights.json disimpan: {cw_path}")

=== CLASS WEIGHTS ===
  B3: count=3466, weight=2.9000
  Organik: count=13744, weight=0.7313
  Recylable: count=12944, weight=0.7765

[OK] class_weights.json disimpan: d:\UGM\Semester 5\Senpro\Project\Senpro\preprocessing_output\class_weights.json


In [12]:
# ============================================================
# RENCANA AUGMENTASI (dicatat di sini, dieksekusi di Colab)
# ============================================================

# CATATAN AUGMENTASI UNTUK COLAB (Bagian B, Tahap 5):
#
# === Augmentasi UMUM (semua kelas, split TRAIN saja) ===
# - RandomHorizontalFlip(p=0.5)
# - RandomVerticalFlip(p=0.2)
# - ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05)
# - RandomResizedCrop(size=384, scale=(0.8, 1.0))  → sebelum SiglipImageProcessor
#
# === Augmentasi KHUSUS B3 (lebih agresif, untuk counter imbalance) ===
# - RandomHorizontalFlip(p=0.5)
# - RandomVerticalFlip(p=0.3)
# - RandomRotation(degrees=30)
# - ColorJitter(brightness=0.4, contrast=0.4, saturation=0.2, hue=0.1)
# - RandomResizedCrop(size=384, scale=(0.7, 1.0))
# - RandomGrayscale(p=0.05)
#
# PENTING: Augmentasi dilakukan SEBELUM SiglipImageProcessor
# (SiglipImageProcessor menangani resize ke 384x384 dan normalisasi ke [-1, 1])
# Augmentasi hanya aktif pada split TRAIN, tidak untuk val/test.
#
# File konfigurasi augmentasi disimpan terpisah agar bisa dibaca di Colab:

aug_config = {
    "note": "Augmentasi dieksekusi on-the-fly di DataLoader Colab, bukan di preprocessing lokal.",
    "apply_only_on_split": "train",
    "augmentasi_umum": [
        {"transform": "RandomHorizontalFlip", "p": 0.5},
        {"transform": "RandomVerticalFlip", "p": 0.2},
        {"transform": "ColorJitter", "brightness": 0.2, "contrast": 0.2, "saturation": 0.1, "hue": 0.05},
        {"transform": "RandomResizedCrop", "size": 384, "scale": [0.8, 1.0]},
    ],
    "augmentasi_B3_ekstra": [
        {"transform": "RandomHorizontalFlip", "p": 0.5},
        {"transform": "RandomVerticalFlip", "p": 0.3},
        {"transform": "RandomRotation", "degrees": 30},
        {"transform": "ColorJitter", "brightness": 0.4, "contrast": 0.4, "saturation": 0.2, "hue": 0.1},
        {"transform": "RandomResizedCrop", "size": 384, "scale": [0.7, 1.0]},
        {"transform": "RandomGrayscale", "p": 0.05},
    ],
    "urutan_pipeline": [
        "1. Load gambar PIL",
        "2. Terapkan augmentasi (jika split=train)",
        "3. SiglipImageProcessor (resize 384x384, normalize ke [-1,1])",
        "4. Output tensor pixel_values untuk model"
    ]
}

aug_path = os.path.join(OUTPUT_DIR, "augmentasi_config.json")
with open(aug_path, 'w', encoding='utf-8') as f:
    json.dump(aug_config, f, indent=2, ensure_ascii=False)
print(f"[OK] augmentasi_config.json disimpan: {aug_path}")

[OK] augmentasi_config.json disimpan: d:\UGM\Semester 5\Senpro\Project\Senpro\preprocessing_output\augmentasi_config.json


---
## TAHAP 4 – Stratified Split Train/Val/Test

In [ ]:
# Gunakan hanya file dengan status 'ok' dari Tahap 2
# rel_path adalah path relatif terhadap DATASET_DIR → portable ke Google Drive

np.random.seed(RANDOM_SEED)

df_split_input = df_clean[['rel_path', 'label']].copy().reset_index(drop=True)

# Split 1: train vs (val + test)
df_train, df_temp = train_test_split(
    df_split_input,
    test_size=(VAL_RATIO + TEST_RATIO),
    stratify=df_split_input['label'],
    random_state=RANDOM_SEED,
)

# Split 2: val vs test dari sisa
val_frac = VAL_RATIO / (VAL_RATIO + TEST_RATIO)  # proporsi val dari temp
df_val, df_test = train_test_split(
    df_temp,
    test_size=(1 - val_frac),
    stratify=df_temp['label'],
    random_state=RANDOM_SEED,
)

# ── Pindahkan near-duplicate lintas split ke TRAIN (hilangkan kebocoran val/test) ──
def _basename(p):
    return str(p).replace('\\', '/').rsplit('/', 1)[-1]

if CROSS_SPLIT_DUP_CSV and os.path.exists(CROSS_SPLIT_DUP_CSV):
    dup = pd.read_csv(CROSS_SPLIT_DUP_CSV)
    # Yang perlu ditarik ke train: anggota di sisi val/test yang kembar dengan train,
    # plus sisi val/test dari pasangan val<->test (pindahkan keduanya ke train agar tak ada bayangan).
    move_names = set()
    for _, r in dup.iterrows():
        if str(r.get('split', '')).lower() in ('val', 'test'):
            move_names.add(_basename(r['path']))
        if str(r.get('match_split', '')).lower() in ('val', 'test'):
            move_names.add(_basename(r['match_path']))
    bn_val, bn_test = df_val['rel_path'].map(_basename), df_test['rel_path'].map(_basename)
    mv_val, mv_test = df_val[bn_val.isin(move_names)], df_test[bn_test.isin(move_names)]
    n_mv = len(mv_val) + len(mv_test)
    if n_mv:
        df_train = pd.concat([df_train, mv_val, mv_test], ignore_index=True)
        df_val   = df_val[~bn_val.isin(move_names)].reset_index(drop=True)
        df_test  = df_test[~bn_test.isin(move_names)].reset_index(drop=True)
        print(f"[OK] Near-dup lintas split: {n_mv} gambar dipindah ke TRAIN "
              f"(val -{len(mv_val)}, test -{len(mv_test)}). Val/test kini bebas bayangan train.")
    else:
        print("[INFO] Tidak ada anggota near-dup lintas split yang cocok di val/test (nama file).")
else:
    print("[SKIP] CROSS_SPLIT_DUP_CSV kosong / tidak ada -> langkah pindah near-dup dilewati.")

# Verifikasi tidak ada overlap
train_paths = set(df_train['rel_path'])
val_paths   = set(df_val['rel_path'])
test_paths  = set(df_test['rel_path'])
assert len(train_paths & val_paths) == 0, "ADA OVERLAP train-val!"
assert len(train_paths & test_paths) == 0, "ADA OVERLAP train-test!"
assert len(val_paths & test_paths) == 0, "ADA OVERLAP val-test!"

print("=== HASIL SPLIT ===")
print(f"  Train : {len(df_train)} ({len(df_train)/len(df_split_input)*100:.1f}%)")
print(f"  Val   : {len(df_val)} ({len(df_val)/len(df_split_input)*100:.1f}%)")
print(f"  Test  : {len(df_test)} ({len(df_test)/len(df_split_input)*100:.1f}%)")
print()

for split_name, df_s in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    print(f"  {split_name} per kelas:")
    for label, cnt in sorted(df_s['label'].value_counts().items()):
        print(f"    {label}: {cnt}")

In [14]:
# Simpan CSV split
# Format: rel_path (relatif terhadap DATASET_DIR), label
# Path menggunakan forward slash agar portabel ke Linux/Colab

def normalize_path_sep(df: pd.DataFrame) -> pd.DataFrame:
    """Ganti backslash dengan forward slash untuk portabilitas."""
    df = df.copy()
    df['rel_path'] = df['rel_path'].str.replace('\\', '/', regex=False)
    return df

df_train_out = normalize_path_sep(df_train)
df_val_out   = normalize_path_sep(df_val)
df_test_out  = normalize_path_sep(df_test)

train_csv = os.path.join(OUTPUT_DIR, "train.csv")
val_csv   = os.path.join(OUTPUT_DIR, "val.csv")
test_csv  = os.path.join(OUTPUT_DIR, "test.csv")

df_train_out.to_csv(train_csv, index=False)
df_val_out.to_csv(val_csv, index=False)
df_test_out.to_csv(test_csv, index=False)

print(f"[OK] train.csv: {train_csv}")
print(f"[OK] val.csv  : {val_csv}")
print(f"[OK] test.csv : {test_csv}")

[OK] train.csv: d:\UGM\Semester 5\Senpro\Project\Senpro\preprocessing_output\train.csv
[OK] val.csv  : d:\UGM\Semester 5\Senpro\Project\Senpro\preprocessing_output\val.csv
[OK] test.csv : d:\UGM\Semester 5\Senpro\Project\Senpro\preprocessing_output\test.csv


---
## RINGKASAN AKHIR & INSTRUKSI UPLOAD KE GOOGLE DRIVE

In [15]:
print("="*60)
print("  PREPROCESSING LOKAL SELESAI")
print("="*60)
print()
print(f"Output disimpan di: {OUTPUT_DIR}")
print()
print("File yang dihasilkan:")
for fname in ["dedup_log.csv", "metadata_kualitas.csv", "class_weights.json",
              "augmentasi_config.json", "train.csv", "val.csv", "test.csv"]:
    fpath = os.path.join(OUTPUT_DIR, fname)
    size  = os.path.getsize(fpath) if os.path.exists(fpath) else 0
    print(f"  [{size:>8,} bytes] {fname}")

print()
print("="*60)
print("  INSTRUKSI UPLOAD KE GOOGLE DRIVE")
print("="*60)
print("""
Yang perlu diunggah ke Google Drive:

  1. Folder Dataset (TANPA subfolder duplicates_removed)
     → Folder: Dataset/0_Recylable, Dataset/1_Organik, Dataset/2_B3
     → SKIP: Dataset/duplicates_removed (tidak perlu)

  2. Folder preprocessing_output (seluruh isinya):
     → train.csv
     → val.csv
     → test.csv
     → class_weights.json
     → augmentasi_config.json
     → metadata_kualitas.csv  (opsional, untuk referensi)
     → dedup_log.csv          (opsional, untuk referensi)

Struktur yang direkomendasikan di Google Drive:
  MyDrive/
  └── Senpro/
      ├── Dataset/
      │   ├── 0_Recylable/
      │   ├── 1_Organik/
      │   └── 2_B3/
      └── preprocessing_output/
          ├── train.csv
          ├── val.csv
          ├── test.csv
          ├── class_weights.json
          └── augmentasi_config.json

Di notebook Colab, set:
  ROOT_DIR = "/content/drive/MyDrive/Senpro"
""")

  PREPROCESSING LOKAL SELESAI

Output disimpan di: d:\UGM\Semester 5\Senpro\Project\Senpro\preprocessing_output

File yang dihasilkan:
  [       2 bytes] dedup_log.csv
  [2,573,104 bytes] metadata_kualitas.csv
  [      72 bytes] class_weights.json
  [   1,415 bytes] augmentasi_config.json
  [ 673,167 bytes] train.csv
  [ 144,159 bytes] val.csv
  [ 144,273 bytes] test.csv

  INSTRUKSI UPLOAD KE GOOGLE DRIVE

Yang perlu diunggah ke Google Drive:

  1. Folder Dataset (TANPA subfolder duplicates_removed)
     → Folder: Dataset/0_Recylable, Dataset/1_Organik, Dataset/2_B3
     → SKIP: Dataset/duplicates_removed (tidak perlu)

  2. Folder preprocessing_output (seluruh isinya):
     → train.csv
     → val.csv
     → test.csv
     → class_weights.json
     → augmentasi_config.json
     → metadata_kualitas.csv  (opsional, untuk referensi)
     → dedup_log.csv          (opsional, untuk referensi)

Struktur yang direkomendasikan di Google Drive:
  MyDrive/
  └── Senpro/
      ├── Dataset/
      │